In [ ]:
# ==========================================================
# Select Design
# ==========================================================
USE_CONVERTER = True
# USE_CONVERTER = False

In [ ]:
# This is the notebook file for Hardware Matrix Layout Conversion FPGA test

from pynq import Overlay, allocate
import numpy as np

# ---- RTL-defined Matrix Parameters (it should be the same iwth config.svh file) -----
SYSTEM_TOP_WIDTH_INPUT      = 8    # Input width
SYSTEM_TOP_FRAC_WIDTH_INPUT = 0
SYSTEM_TOP_WIDTH_WEIGHT     = 8    # Weight width
SYSTEM_FRAC_WIDTH_WEIGHT    = 0
SYSTEM_TOP_WIDTH_FINAL      = 8    # Output width
SYSTEM_FRAC_WIDTH_FINAL     = 0
I_MATRIX_DIMENSION          = 32    # Input matrix
INNER_MATRIX_DIMENSION      = 32
W_MATRIX_DIMENSION          = 32    # Weight matrix
SYSTEM_NUM_CORES_A          = 2     # Number of cores A
SYSTEM_TOTAL_MODULES        = 2     # Number of cores B
TOP_BLOCK_SIZE              = 2
TOP_CHUNK_SIZE              = 4

In [ ]:
# --- FPGA and Design Parameters ---
if USE_CONVERTER:
    DMA_INPUT_WORD_BITS = (SYSTEM_TOP_WIDTH_INPUT * INNER_MATRIX_DIMENSION)
else:
    DMA_INPUT_WORD_BITS = (SYSTEM_TOP_WIDTH_INPUT * TOP_CHUNK_SIZE * SYSTEM_NUM_CORES_A)

IN_DATA_UNIT_BITS   = SYSTEM_TOP_WIDTH_INPUT  # uint16, same with SYSTEM_TOP_WIDTH_INPUT
WORDS_PER_INPUT     = DMA_INPUT_WORD_BITS // IN_DATA_UNIT_BITS

DMA_OUTPUT_WORD_BITS           = SYSTEM_TOP_WIDTH_FINAL * TOP_CHUNK_SIZE * SYSTEM_NUM_CORES_A * SYSTEM_TOTAL_MODULES 
OUT_DATA_UNIT_BITS          = SYSTEM_TOP_WIDTH_FINAL  # uint16, SYSTEM_TOP_WIDTH_FINAL
WORDS_PER_OUTPUT            = DMA_OUTPUT_WORD_BITS // OUT_DATA_UNIT_BITS


if OUT_DATA_UNIT_BITS == 8:
    out_dtype = np.uint8
elif OUT_DATA_UNIT_BITS == 16:
    out_dtype = np.uint16
elif OUT_DATA_UNIT_BITS == 32:
    out_dtype = np.uint32
elif OUT_DATA_UNIT_BITS == 64:
    out_dtype = np.uint64
else:
    raise ValueError("Unsupported output width")

In [ ]:
# --- Derived output size ---
ROWS                = I_MATRIX_DIMENSION // (TOP_BLOCK_SIZE * SYSTEM_NUM_CORES_A)
COLS                = W_MATRIX_DIMENSION // (TOP_BLOCK_SIZE * SYSTEM_TOTAL_MODULES)              
TOTAL_OUTPUT_WORDS  = (ROWS * COLS)
OUTPUT_BUFFER_WORDS = TOTAL_OUTPUT_WORDS * WORDS_PER_OUTPUT

# File paths
INPUT_MEM_FILE              = "i.mem"
WEIGHT_MEM_FILE             = "w.mem"
OUTPUT_MEM_FILE             = "o.mem"

# --- Load Overlay ---
if USE_CONVERTER:
    BITFILE = "/home/xilinx/jupyter_notebooks/Matrix_Multiplier/design_converter.bit"
else:
    BITFILE = "/home/xilinx/jupyter_notebooks/Matrix_Multiplier/design_WITHOUT_ converter.bit"

overlay = Overlay(BITFILE)
print("Overlay loaded.")

In [ ]:
dma_i   = overlay.axi_dma_0
dma_w   = overlay.axi_dma_1
dma_o   = overlay.axi_dma_2

In [ ]:
# --- Load HEX .mem File ---
def load_mem_file(filename, word_bits, data_unit_bits):
    with open(filename, "r") as f:
        hex_data = f.read().replace("\n", "").strip()

    word_hex_len = word_bits // 4
    unit_hex_len = data_unit_bits // 4

    chunks = [
        hex_data[i:i+word_hex_len]
        for i in range(0, len(hex_data), word_hex_len)
    ]

    data = []

    for word in chunks:
        word = word.zfill(word_hex_len)

        # Extract from LSB to MSB
        for i in range(0, len(word), unit_hex_len):
            start = len(word) - unit_hex_len - i
            end   = len(word) - i
            data.append(int(word[start:end], 16))

    if data_unit_bits == 8:
        dtype = np.uint8
    elif data_unit_bits == 16:
        dtype = np.uint16
    elif data_unit_bits == 32:
        dtype = np.uint32
    elif data_unit_bits == 64:
        dtype = np.uint64
    else:
        raise ValueError(f"Unsupported DATA_UNIT_BITS = {data_unit_bits}")

    return np.array(data, dtype=dtype)

In [ ]:
# --- Load Inputs ---
input_data = load_mem_file(INPUT_MEM_FILE, DMA_INPUT_WORD_BITS, IN_DATA_UNIT_BITS)
weight_data = load_mem_file(WEIGHT_MEM_FILE, DMA_INPUT_WORD_BITS, IN_DATA_UNIT_BITS)

In [ ]:
# --- Allocate Buffers ---
input_buffer = allocate(
    shape=input_data.shape,
    dtype=input_data.dtype
)

weight_buffer = allocate(
    shape=weight_data.shape,
    dtype=weight_data.dtype
)

np.copyto(input_buffer, input_data)
np.copyto(weight_buffer, weight_data)

input_buffer.flush()
weight_buffer.flush()

output_buffer = allocate(
    shape=(OUTPUT_BUFFER_WORDS,),
    dtype=out_dtype
)

In [ ]:
print(f"Input buffer size: {input_buffer.nbytes} bytes ({input_buffer.shape[0]} words)")
print(f"Weight buffer size: {weight_buffer.nbytes} bytes ({weight_buffer.shape[0]} words)")
print(f"Output buffer size: {output_buffer.nbytes} bytes ({output_buffer.shape[0]} words)")

In [ ]:
def print_hex_chunks(data_array, bits_per_word, label):
    bits_per_element = data_array.dtype.itemsize * 8
    words_per_line = bits_per_word // bits_per_element
    print(f"\n{label} contents ({bits_per_word}-bit words):")
    reshaped = data_array.reshape((-1, words_per_line))
    for i, line in enumerate(reshaped):
        hex_digits = bits_per_element // 4
        hex_word = ''.join(
            f"{x:0{hex_digits}x}"
            for x in reversed(line)
        )
        print(f"{label} {i}: {hex_word}")

print_hex_chunks(input_buffer, DMA_INPUT_WORD_BITS, "Input 0")
print_hex_chunks(weight_buffer, DMA_INPUT_WORD_BITS, "Input 1")

In [ ]:
# --- Start Transfers ---
dma_o.recvchannel.transfer(output_buffer)
dma_i.sendchannel.transfer(input_buffer)
dma_w.sendchannel.transfer(weight_buffer)

dma_i.sendchannel.wait()
dma_w.sendchannel.wait()
dma_o.recvchannel.wait()

In [ ]:
output_buffer.invalidate()
print_hex_chunks(output_buffer, DMA_OUTPUT_WORD_BITS, "Output")

In [ ]:
# --- Reconstruct Output Data ---
output_chunks = output_buffer.reshape((-1, WORDS_PER_OUTPUT))
hex_digits = output_buffer.dtype.itemsize * 2

output_words = [
    ''.join(
        f"{x:0{hex_digits}x}"
        for x in reversed(chunk)
    )
    for chunk in output_chunks
]

print("Output received (128-bit hex):")
for i, word in enumerate(output_words):
    print(f"Output {i}: {word}")


In [ ]:
with open(OUTPUT_MEM_FILE, "w") as f:
    for word in output_words:
        f.write(word + "\n")

print(f"Saved output to {OUTPUT_MEM_FILE}")

In [ ]:
# Delete buffer to prevent memory leak
del input_buffer, weight_buffer, output_buffer